## Integration of Postgres with Pandas
Pandas is a powerful Python Data Library which is used to process as well as analyze the data. It have robust APIs to work with databases.

Here are the steps involved to use Pandas to work with databases like Postgres.
* We need to make sure `pandas`, `psycopg2-binary` as well as `sqlalchemy` installed using `pip`.
* Pandas uses `sqlalchemy` to interact with database tables based on the connection url.
* `sqlalchemy` is the most popular ORM to hide the complexity of connecting to the databases using our applications.
* Here are the examples using Pandas. 
  * Read from file, Process and Write to Postgres Database Table using Pandas. 
  * Read from Postgres Database Table using Pandas for the validation.

In [ ]:
!pip install psycopg2-binary

In [ ]:
!pip install pandas

In [ ]:
!pip install sqlalchemy

In [ ]:
import pandas as pd

In [ ]:
columns = ['order_id', 'order_date', 'order_customer_id', 'order_status']

In [ ]:
import json

In [ ]:
schemas = json.load(open('../../data/retail_db/schemas.json'))

In [ ]:
sorted(schemas['orders'], key=lambda col: col['column_position'])

In [ ]:
columns = [col['column_name'] for col in sorted(schemas['orders'], key=lambda col: col['column_position'])]

In [ ]:
columns

In [ ]:
pd.read_csv?

In [ ]:
orders = pd.read_csv('../../data/retail_db/orders/part-00000', names=columns)

In [ ]:
orders

In [ ]:
type(orders)

In [ ]:
orders.shape

In [ ]:
daily_status_count = orders. \
    groupby(['order_date', 'order_status'])['order_id']. \
    agg(order_count='count'). \
    reset_index()

In [ ]:
daily_status_count

In [ ]:
daily_status_count.to_sql?

In [ ]:
host = '104.197.141.244'
port = 5432
database = 'itversity_retail_db'
user = 'itversity_retail_user'
password = 'itversity'

In [ ]:
conn_uri = f'postgresql://{user}:{password}@{host}:{port}/{database}'

In [ ]:
daily_status_count.to_sql(
    'daily_status_count',
    conn_uri,
    if_exists='replace',
    index=False
)

In [ ]:
pd.read_sql?

In [ ]:
dsc = pd.read_sql(
    'daily_status_count',
    conn_uri
)

In [ ]:
dsc.shape

In [ ]:
dsc

In [ ]:
dsc = pd.read_sql(
    '''
        SELECT order_status, sum(order_count) AS order_count FROM daily_status_count
        GROUP BY 1
        ORDER BY 2 DESC
    ''',
    conn_uri
)

In [ ]:
dsc